# Phase 0: DB Generation — Superstore Dataset

**Notebook Objective:** 
> This notebook audits the raw Superstore CSV file and then load the cleaned data into a SQLite database for later use in the subsequent project phases.

---

**Contents:**
1. Setup
2. Data Quality Check-up
3. Apply Data Quality fixes
4. Load the cleaned data into SQLite
5. Cleaned Database Validation
6. Data Quality Summary

---
## 1. Setup

In [13]:
# Import libraries
import re
import sqlite3
from pathlib import Path

import pandas as pd

# Define paths
ROOT     = Path().resolve().parent
CSV      = ROOT / "data" / "Sample - Superstore.csv"
DB       = ROOT / "data" / "superstore.db"
SQL_PATH = ROOT / "sql"

print(f"CSV       : {CSV}  (exists: {CSV.exists()})")
print(f"DB        : {DB}  (exists: {DB.exists()})")
print(f"SQL_PATH  : {SQL_PATH}  (exists: {SQL_PATH.exists()})")


def run_sql_file(filepath):
    """Read a .sql file, strip comments, and return a list of DataFrames — one per statement."""
    with open(filepath) as f:
        content = f.read()
    content = re.sub(r'--[^\n]*', '', content)          # strip line comments
    statements = [s.strip() for s in content.split(';') if s.strip()]
    return [pd.read_sql_query(s, conn) for s in statements]

CSV       : /Users/josuxx/ai_space/workspace/01_projects/superstore-sales-analysis/data/Sample - Superstore.csv  (exists: True)
DB        : /Users/josuxx/ai_space/workspace/01_projects/superstore-sales-analysis/data/superstore.db  (exists: True)
SQL_PATH  : /Users/josuxx/ai_space/workspace/01_projects/superstore-sales-analysis/sql  (exists: True)


---
## 2. Data Quality Check-up

Before generating the database from the CSV (the raw data), the data is audited for completeness and consistency.
The dataset is a public Kaggle sample, so it is mostly clean. However, verifying is cheaper than assuming.

The checks include:

- shape and column types
- missing values
- date-range sanity
- duplicates
- cleaned database validation

---
### 2.1 Shape & Column Types

In [14]:
#Read the raw dataset (CSV file)
df = pd.read_csv(CSV, encoding="latin-1")

print(f'Shape (raw CSV): {df.shape[0]:,} rows x {df.shape[1]} columns')

print('\nColumn types:')
display(df.dtypes.to_frame('dtype'))

print('\nFirst 5 rows:')
display(df.head())

Shape (raw CSV): 9,994 rows x 21 columns

Column types:


,dtype
Row ID,int64
Order ID,str
Order Date,str
Ship Date,str
Ship Mode,str
Customer ID,str
Customer Name,str
Segment,str
Country,str
City,str



First 5 rows:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


---
### 2.2 Nulls (Missing Values)

In [15]:
print('\nMissing values per column:')
display(df.isna().sum().to_frame('nulls'))


Missing values per column:


,nulls
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


---
### 2.3 Date Range Sanity (Order ≤ Ship; range)

In [16]:
_order = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
_ship  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')

print(f"Order Date range: {_order.min().date()} -> {_order.max().date()}")
print(f"Ship Date  range: {_ship.min().date()} -> {_ship.max().date()}")

violations = (_ship < _order).sum()
print(f'\nShip Date earlier than Order Date: {violations} row(s)')

Order Date range: 2014-01-03 -> 2017-12-30
Ship Date  range: 2014-01-07 -> 2018-01-05

Ship Date earlier than Order Date: 0 row(s)


---
### 2.4 Duplicate Audit

The raw CSV is checked for duplicates at two levels:

1. **Fully duplicated rows:** exact matches across all columns, including `Row ID`.
2. **Duplicated business records:** rows that match across all columns except `Row ID`.

Because `Row ID` is a unique technical identifier, a duplicated business record may not appear as a fully duplicated row if only the `Row ID` differs.

In [17]:
print(f'Fully duplicated rows in raw CSV: {df.duplicated().sum()}')
print(f'Row ID is unique in raw CSV: {df["Row ID"].is_unique}')

duplicate_check_cols = [
    'Order ID',
    'Order Date',
    'Ship Date',
    'Ship Mode',
    'Customer ID',
    'Customer Name',
    'Segment',
    'Country',
    'City',
    'State',
    'Postal Code',
    'Region',
    'Product ID',
    'Category',
    'Sub-Category',
    'Product Name',
    'Sales',
    'Quantity',
    'Discount',
    'Profit'
]

duplicate_rows_excluding_row_id = df[
    df.duplicated(subset=duplicate_check_cols, keep=False)
]

print(
    f'\nDuplicated rows excluding Row ID in raw CSV: '
    f'{len(duplicate_rows_excluding_row_id)}'
)

display(
    duplicate_rows_excluding_row_id[
        ['Row ID'] + duplicate_check_cols
    ].sort_values(['Order ID', 'Product ID', 'Row ID'])
)

Fully duplicated rows in raw CSV: 0
Row ID is unique in raw CSV: True

Duplicated rows excluding Row ID in raw CSV: 2


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
3405,3406,US-2014-150119,4/23/2014,4/27/2014,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588
3406,3407,US-2014-150119,4/23/2014,4/27/2014,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588


---
## 3. Apply data quality fixes

Three transformations are applied before writing to SQLite. The first fix addresses a duplicate row, while the second and third fixes standardize data types to make downstream SQL queries cleaner and more reliable:

### Fix 1 — Remove confirmed duplicate row

- Row IDs 3406 and 3407 were identified in section 2 as a confirmed duplicate line item with the same attributes
- Row ID 3407 is removed before loading to keep the SQLite source deduplicated

In [18]:
before = len(df)
df = df[df["Row ID"] != 3407].reset_index(drop=True)
print(f"Dropped 1 duplicate row (Row ID 3407). Rows now: {len(df):,} (was {before:,})")

Dropped 1 duplicate row (Row ID 3407). Rows now: 9,993 (was 9,994)


### Fix 2 — Normalize dates to ISO format (`YYYY-MM-DD`)

- `Order Date` and `Ship Date` arrive in the raw CSV as `M/D/YYYY`
- They are converted to ISO format (`YYYY-MM-DD`) to make downstream SQL date queries cleaner and easier to maintain
- This enables simpler use of SQLite date functions such as `strftime()`, `julianday()`, and `DATE()`

In [19]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y').dt.strftime('%Y-%m-%d')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y').dt.strftime('%Y-%m-%d')

### Fix 3 — Convert `Postal Code` to zero-padded text

- `Postal Code` is read as an integer by default, which silently drops the leading zero from Northeast US zip codes, e.g. `02101` → `2101`.

- Storing it as a zero-padded 5-character string preserves the correct format.

In [20]:
df['Postal Code'] = df['Postal Code'].astype(str).str.zfill(5)

print(f"Order Date sample:  {df['Order Date'].iloc[0]}  (dtype: {df['Order Date'].dtype})")
print(f"Ship Date sample:   {df['Ship Date'].iloc[0]}  (dtype: {df['Ship Date'].dtype})")
print(f"Postal Code sample: {df['Postal Code'].iloc[0]}  (dtype: {df['Postal Code'].dtype})")

Order Date sample:  2016-11-08  (dtype: str)
Ship Date sample:   2016-11-11  (dtype: str)
Postal Code sample: 42420  (dtype: str)


---
## 4. Load cleaned data into SQLite

In [21]:
conn = sqlite3.connect(DB)
df.to_sql("orders", conn, if_exists="replace", index=False)
conn.close()
print(f"Done. {len(df):,} rows written to {DB.name} -> table 'orders'")

Done. 9,993 rows written to superstore.db -> table 'orders'


---
## 5. Cleaned Database Validation

The cleaned SQLite database is validated to confirm that the quality fixes from section 3 were applied correctly and that the database is ready for the next phase.

The validation queries are defined in [`sql/00_db_validation.sql`](../sql/00_db_validation.sql) and run together below:

- `row_count` — total rows in `orders`
- `sample_fields` — sample of `Order Date`, `Ship Date`, `Postal Code` (confirms Fix 2 and Fix 3)
- `removed_duplicate` — confirms the duplicate row identified in section 2 is gone
- `integrity_summary` — totals, date range, and nulls in key numeric fields
- `repeated_pairs` — informational: `(Order ID, Product ID)` pairs that appear more than once (these are legitimate multi-line orders, not duplicates)

In [22]:
conn = sqlite3.connect(DB)

(
    row_count,
    sample_fields,
    removed_duplicate,
    integrity_summary,
    repeated_pairs,
) = run_sql_file(SQL_PATH / "00_db_validation.sql")

conn.close()

print("Row count:")
display(row_count)

print("\nSample of key columns (Order Date, Ship Date, Postal Code):")
display(sample_fields)

print("\nRows remaining for the previously identified duplicate (Order ID + Product ID):")
display(removed_duplicate)

print("\nIntegrity summary (totals, date range, nulls):")
display(integrity_summary)

print(f"\nRepeated (Order ID, Product ID) pairs in cleaned DB: {len(repeated_pairs)}")
display(repeated_pairs)

Row count:


,rows
0,9993



Sample of key columns (Order Date, Ship Date, Postal Code):


,Order Date,Ship Date,Postal Code
0,2016-11-08,2016-11-11,42420
1,2016-11-08,2016-11-11,42420
2,2016-06-12,2016-06-16,90036



Rows remaining for the previously identified duplicate (Order ID + Product ID):


,Row ID,Order ID,Product ID,Quantity,Sales,Discount,Profit
0,3406,US-2014-150119,FUR-CH-10002965,2,281.372,0.3,-12.0588



Integrity summary (totals, date range, nulls):


,total_rows,unique_row_ids,removed_duplicate_present,min_order_date,max_order_date,null_sales,null_profit,null_discount
0,9993,9993,0,2014-01-03,2017-12-30,0,0,0



Repeated (Order ID, Product ID) pairs in cleaned DB: 7


,Order ID,Product ID,line_count
0,CA-2015-103135,OFF-BI-10000069,2
1,CA-2016-129714,OFF-PA-10001970,2
2,CA-2016-137043,FUR-FU-10003664,2
3,CA-2016-140571,OFF-PA-10001954,2
4,CA-2017-118017,TEC-AC-10002006,2
5,CA-2017-152912,OFF-ST-10003208,2
6,US-2016-123750,TEC-AC-10004659,2


---
## 6. Data Quality Summary

| Check | Result |
|---|---|
| Raw CSV shape | 9,994 rows x 21 columns |
| Missing values in raw CSV | 0 across all columns |
| Order Date range | 2014-01-03 -> 2017-12-30 |
| Ship Date earlier than Order Date | 0 violations |
| Row ID uniqueness in raw CSV | Unique |
| Fully duplicated rows in raw CSV | 0 |
| Duplicated rows excluding `Row ID` in raw CSV | 2 rows / 1 duplicated record |
| Cleaning action | Row ID 3407 removed before writing to SQLite |
| Cleaned SQLite database shape | 9,993 rows |
| Removed duplicate present in cleaned database | No |
| Duplicated rows excluding `Row ID` in cleaned database | 0 |
| Nulls in key numeric fields in cleaned database | 0 in Sales, Profit, and Discount |